# AI Coursework — Training Pipeline

**Run cells top to bottom in order.**

| Cell | Purpose | Run |
|------|---------|-----|
| 1 | Mount Google Drive | Every session |
| 2 | Clone / update repo + install packages | Every session |
| 3 | Symlink Drive paths into repo | Every session |
| 4 | **Pre-compute embeddings** (dropdown) | Once per combination |
| 5 | **Train + Evaluate** (dropdown) | Once per experiment |
| 6 | View results | Anytime |

---
### Before you start
1. Go to **Runtime → Change runtime type → T4 GPU → Save**
2. The team Drive folder must be added to your own Drive:
   - Open the [team Drive folder](https://drive.google.com/drive/folders/1EL_dn-MavuF7DJ9_W4hN0_uDsxWV04HE?usp=sharing)
   - Click the folder name at the top → **Add shortcut to Drive** → **My Drive** → Add
   - After this, the folder appears in your Drive as a shortcut

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ── Cell 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# ── Cell 2: Clone / update repo + install dependencies ────────────────────────
import os

REPO_URL = 'https://github.com/Rylie-KIM/uob-ds-intro-to-ai-final-cw-2026.git'
REPO_DIR = '/content/repo'

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -r {REPO_DIR}/requirements.txt -q
print('Done.')

Cloning into '/content/repo'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: '/content/repo'
/content
ERROR: Could not open requirements file: [Errno 2] No such file or directory: '/content/repo/requirements.txt'
Done.


In [ ]:
# ── Cell 3: Symlink Drive paths into repo ─────────────────────────────────────
#
# The team Drive folder is located by its folder ID — no manual path editing needed.
# Prerequisite: the team folder shortcut must be added to your MyDrive (see Cell 0 note).

import os, pathlib, subprocess

REPO_DIR   = '/content/repo'
FOLDER_ID  = '1EL_dn-MavuF7DJ9_W4hN0_uDsxWV04HE'

# ── Auto-detect the Drive folder path by its ID ───────────────────────────────
result = subprocess.run(
    ['find', '/content/drive', '-name', f'.shortcut-targets-by-id'],
    capture_output=True, text=True
)

# Try shortcut path first (most common case)
shortcut_path = f'/content/drive/.shortcut-targets-by-id/{FOLDER_ID}'
direct_path_candidates = [
    shortcut_path,
    f'/content/drive/MyDrive/ai-cw-2026',          # fallback: manually placed folder
]

DRIVE_BASE = None
for candidate in direct_path_candidates:
    if os.path.exists(candidate):
        DRIVE_BASE = candidate
        print(f'Found team Drive folder at: {DRIVE_BASE}')
        break

if DRIVE_BASE is None:
    raise FileNotFoundError(
        '\nTeam Drive folder not found.\n'
        'Please add the shortcut to your Drive first:\n'
        '  https://drive.google.com/drive/folders/1EL_dn-MavuF7DJ9_W4hN0_uDsxWV04HE\n'
        '  → click the folder name at the top → Add shortcut to Drive → My Drive → Add'
    )

# ── Create symlinks ───────────────────────────────────────────────────────────
links = {
    f'{DRIVE_BASE}/images/type-a':  f'{REPO_DIR}/src/data/images/type-a',
    f'{DRIVE_BASE}/images/type-b':  f'{REPO_DIR}/src/data/images/type-b',
    f'{DRIVE_BASE}/images/type-c':  f'{REPO_DIR}/src/data/images/type-c',
    f'{DRIVE_BASE}/embeddings':     f'{REPO_DIR}/results/embeddings',
    f'{DRIVE_BASE}/checkpoints':    f'{REPO_DIR}/results/checkpoints',
}

for src, dst in links.items():
    pathlib.Path(src).mkdir(parents=True, exist_ok=True)
    dst_path = pathlib.Path(dst)
    if not dst_path.exists() and not dst_path.is_symlink():
        dst_path.parent.mkdir(parents=True, exist_ok=True)
        os.symlink(src, dst)
        print(f'linked : {dst}')
    else:
        print(f'ok     : {dst}')

print('\nAll paths ready.')

---
## Phase 1 — Pre-compute Embeddings

- Run **once per (dataset, embedding) combination**.
- If the `.pt` file already exists on Drive, this step is automatically skipped.
- Select only the dataset you are responsible for and the embeddings you plan to use.

In [ ]:
# ── Cell 4: Phase 1 — Pre-compute embeddings ──────────────────────────────────
import ipywidgets as widgets
from IPython.display import display

dataset_w = widgets.SelectMultiple(
    options=[
        ('Type-A  (shapes + colour descriptions)', 'a'),
        ('Type-B  (coloured MNIST numbers)',       'b'),
        ('Type-C  (tic-tac-toe boards)',           'c'),
    ],
    value=['b'],
    description='Dataset',
    layout=widgets.Layout(width='380px', height='90px'),
)
embedding_w = widgets.SelectMultiple(
    options=[
        ('SBERT     all-MiniLM-L6-v2   (384-dim)', 'sbert'),
        ('BERT      bert-base-uncased  (768-dim)',  'bert'),
        ('TinyBERT  4L-312D            (312-dim)',  'tinybert'),
    ],
    value=['sbert'],
    description='Embedding',
    layout=widgets.Layout(width='380px', height='90px'),
)
run_btn_p1 = widgets.Button(
    description='Run Phase 1',
    button_style='primary',
    layout=widgets.Layout(width='160px', margin='12px 0 0 0'),
)
out_p1 = widgets.Output()
hint = widgets.HTML('<i style="color:#888;font-size:12px">Ctrl+click to select multiple items</i>')

display(widgets.HBox([dataset_w, embedding_w]), hint, run_btn_p1, out_p1)

def on_run_p1(b):
    out_p1.clear_output()
    datasets   = list(dataset_w.value)
    embeddings = list(embedding_w.value)
    with out_p1:
        if not datasets or not embeddings:
            print('Select at least one dataset and one embedding.')
            return
        for d in datasets:
            for e in embeddings:
                print(f'\n── dataset={d}  embedding={e} ──')
                !python src/embeddings/precompute_embeddings.py --dataset {d} --embedding {e}
        print('\nPhase 1 complete.')

run_btn_p1.on_click(on_run_p1)

---
## Phase 2 — Train + Evaluate

- Only select combinations whose `.pt` embedding file already exists on Drive (Phase 1 done).
- Multiple models and embeddings can be selected — they will run sequentially.

In [ ]:
# ── Cell 5: Phase 2 — Train + Evaluate ────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display

dataset_w2 = widgets.Dropdown(
    options=[
        ('Type-A  (shapes + colour descriptions)', 'a'),
        ('Type-B  (coloured MNIST numbers)',       'b'),
        ('Type-C  (tic-tac-toe boards)',           'c'),
    ],
    value='b',
    description='Dataset',
    layout=widgets.Layout(width='380px'),
)
model_w = widgets.SelectMultiple(
    options=[
        ('AlexNet128', 'alexnet'),
        ('CNN',        'cnn'),
    ],
    value=['alexnet'],
    description='Model',
    layout=widgets.Layout(width='200px', height='68px'),
)
embedding_w2 = widgets.SelectMultiple(
    options=[
        ('SBERT    (384)', 'sbert'),
        ('BERT     (768)', 'bert'),
        ('TinyBERT (312)', 'tinybert'),
    ],
    value=['sbert'],
    description='Embedding',
    layout=widgets.Layout(width='240px', height='90px'),
)
epochs_w = widgets.IntSlider(
    value=30, min=1, max=100, step=1,
    description='Epochs',
    layout=widgets.Layout(width='400px'),
)
run_btn_p2 = widgets.Button(
    description='Run Training',
    button_style='success',
    layout=widgets.Layout(width='160px', margin='12px 0 0 0'),
)
out_p2 = widgets.Output()
hint2 = widgets.HTML('<i style="color:#888;font-size:12px">Ctrl+click to select multiple models / embeddings</i>')

display(dataset_w2, widgets.HBox([model_w, embedding_w2]), epochs_w, hint2, run_btn_p2, out_p2)

def on_run_p2(b):
    out_p2.clear_output()
    dataset    = dataset_w2.value
    models     = list(model_w.value)
    embeddings = list(embedding_w2.value)
    epochs     = epochs_w.value
    pipeline   = f'src/pipelines/pipeline_{dataset}.py'
    with out_p2:
        if not models or not embeddings:
            print('Select at least one model and one embedding.')
            return
        total = len(models) * len(embeddings)
        done  = 0
        for m in models:
            for e in embeddings:
                done += 1
                print(f'\n[{done}/{total}]  dataset={dataset}  model={m}  embedding={e}  epochs={epochs}')
                print('-' * 60)
                !python {pipeline} --model {m} --embedding {e} --epochs {epochs}
        print('\nPhase 2 complete.')

run_btn_p2.on_click(on_run_p2)

---
## Results

In [ ]:
# ── Cell 6: View results summary ──────────────────────────────────────────────
import pandas as pd
from pathlib import Path

summary_path = Path('results/metrics/results_summary.csv')

if not summary_path.exists():
    print('No results yet. Run Phase 2 first.')
else:
    df = pd.read_csv(summary_path)
    df = df.sort_values('top_1_acc', ascending=False).reset_index(drop=True)
    for col in ['top_1_acc', 'top_5_acc', 'mean_cosine_sim']:
        if col in df.columns:
            df[col] = df[col].map(lambda x: f'{x:.4f}')
    display(df)